# Polynomial Regression

**Topic:** Supervised Learning — Regression

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how polynomial regression fits curves by transforming input features before applying linear regression
- **Explain** why increasing polynomial degree improves training fit but eventually causes overfitting
- **Interpret** a train vs. validation error curve to choose the right degree

> **Tip:** In the **Basis Function Explorer**, watch how each added degree contributes a new curve shape and a new learned weight. Then in the **Overfitting Boundary Widget**, drag the degree slider slowly from 1 to 12 — training fit keeps improving, but past a certain point the fitted curve starts swinging wildly near the edges of the data and validation performance collapses. That's the bias-variance tradeoff made visual.

---
## How we got here

Polynomial regression is linear regression applied to expanded features:

- **[supervised/02_multiple_linear_regression.ipynb](02_multiple_linear_regression.ipynb)** — the matrix formulation and normal equations you need; polynomial regression is multiple regression on engineered inputs
- **[ml_concepts/05_overfitting_and_underfitting.ipynb](../ml_concepts/05_overfitting_and_underfitting.ipynb)** — the core tension this notebook makes visual: low degree underfits, high degree overfits
- **[feature_engineering/05_polynomial_features.ipynb](../feature_engineering/05_polynomial_features.ipynb)** — how sklearn's PolynomialFeatures transformer builds the expanded feature matrix

---
## Why this matters for data science

Many real-world relationships are not straight lines. Sales growth accelerates before leveling off. Drug efficacy peaks at a dose and then declines. Housing prices grow faster at the high end than the low end. Polynomial regression handles these curves while staying within the linear model family — the transformed features are nonlinear, but the coefficients are still fit linearly.

It also introduces one of the most important concepts in all of machine learning: the degree hyperparameter directly controls model complexity, and choosing it requires cross-validation. Everything you learn about selecting degree here generalizes to every complexity-controlling hyperparameter you will tune later.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Polynomial regression sits in the **middle of the spectrum**, drifting right as degree increases. At degree 1 it is identical to multiple linear regression — fully interpretable. At degree 5 it is fitting a curve with hundreds of interaction terms, and the individual coefficients become very hard to interpret.

The practical lesson is that model complexity is a dial, not a binary. You choose how far right you move by choosing degree, and cross-validation tells you when you have gone too far.

---
## How it learns

Imagine you are trying to fit a curved line through a scatter of points. A straight line leaves a clear curved pattern in the residuals — it is systematically too low in the middle and too high at the ends. You need more flexibility.

The trick is to create new features from the ones you already have. For a single feature $x$, you generate $x^2$, $x^3$, and so on. Then you hand this expanded feature matrix to ordinary linear regression. The algorithm still learns a weighted sum — but now the weights apply to powers of $x$, so the result can be any polynomial curve.

The catch is that high-degree polynomials have enormous flexibility. A degree-10 polynomial can weave through every training point almost exactly — but it will look wildly wrong on new data. That is overfitting, and it is the central tension this notebook explores.

---
## The math behind it

Polynomial regression applies a **feature map** $\phi$ to the inputs before linear regression:

$$\phi(x) = [1,\; x,\; x^2,\; x^3,\; \ldots,\; x^d]$$

For a single feature, this gives a degree-$d$ polynomial. The model is still linear in the coefficients:

$$\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \cdots + \theta_d x^d = \boldsymbol{\theta}^\top \phi(x)$$

For $p$ features, PolynomialFeatures also adds **interaction terms** ($x_1 x_2$, $x_1^2 x_2$, etc.), making the feature count grow as $\binom{p + d}{d}$.

**Loss function** — same MSE as linear regression, minimized via the normal equations on the expanded feature matrix:

$$J(\boldsymbol{\theta}) = \frac{1}{n} \sum_{i=1}^{n} \left(\boldsymbol{\theta}^\top \phi(\mathbf{x}_i) - y_i\right)^2$$

**Optimization criterion:** minimize MSE. The degree $d$ is a hyperparameter you choose using cross-validation — the normal equations find the best coefficients for any fixed $d$, but cannot choose $d$ themselves.

---
## Try it yourself

Two widgets below: the **Basis Function Explorer** shows what each added polynomial degree contributes, and the **Overfitting Boundary Widget** shows what happens to training and validation performance as degree climbs.

### Basis Function Explorer

In [ ]:
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

np.random.seed(42)
_n_synth_w1 = 60
_x_synth_w1 = np.linspace(-1, 1, _n_synth_w1)
_true_w1 = 0.5 - 1.2 * _x_synth_w1 + 2.0 * _x_synth_w1 ** 3
_y_synth_w1 = _true_w1 + np.random.normal(0, 0.15, _n_synth_w1)

out_basis = Output()

degree_sl_w1 = IntSlider(value=3, min=1, max=8, step=1,
    description="Degree:", style={"description_width": "70px"},
    layout=widgets.Layout(width="380px"))

_BASIS_COLORS = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"],
                  PALETTE["muted"], "#AE3EC9", "#0CA678", "#F08C00", "#1864AB"]

def render_basis(change=None):
    d = degree_sl_w1.value
    pf = PolynomialFeatures(degree=d, include_bias=False)
    Xp = pf.fit_transform(_x_synth_w1.reshape(-1, 1))
    lr = LinearRegression().fit(Xp, _y_synth_w1)

    x_line = np.linspace(-1, 1, 200)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "Basis functions x¹ … xᵈ", "Learned coefficients"))

    for k in range(1, d + 1):
        fig.add_trace(go.Scatter(
            x=x_line, y=x_line ** k, mode="lines",
            line=dict(color=_BASIS_COLORS[(k - 1) % len(_BASIS_COLORS)], width=2),
            name=f"x^{k}",
        ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=[f"x^{k}" for k in range(1, d + 1)], y=lr.coef_,
        marker_color=PALETTE["primary"], showlegend=False,
    ), row=1, col=2)

    fig.update_layout(
        title=dict(text=f"Degree {d} — {d} basis function(s), {d} learned weight(s)",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
        height=420, paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"]),
    )
    with out_basis:
        clear_output(wait=True)
        fig.show()

degree_sl_w1.observe(render_basis, names="value")
display(VBox([degree_sl_w1, out_basis]))
render_basis()


### Overfitting Boundary Widget

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
import numpy as np

np.random.seed(42)
_h_w2b = fetch_california_housing(as_frame=True)
_x_all_w2b = _h_w2b.data["MedInc"].values
_y_all_w2b = _h_w2b.target.values
# A small subsample (n=100) is deliberate: with the full ~20k rows a single-feature
# degree-12 polynomial can't overfit at all (only 12 parameters vs. thousands of rows).
# n=100 puts model flexibility and data volume in the same ballpark, so the
# overfitting behavior this widget is meant to teach actually shows up.
_idx_w2b = np.random.RandomState(42).choice(len(_x_all_w2b), 100, replace=False)
_x_w2b, _y_w2b = _x_all_w2b[_idx_w2b], _y_all_w2b[_idx_w2b]
_xtr_w2b, _xva_w2b, _ytr_w2b, _yva_w2b = train_test_split(
    _x_w2b, _y_w2b, test_size=0.3, random_state=42)

out_boundary = Output()

degree_sl_w2 = IntSlider(value=2, min=1, max=12, step=1,
    description="Degree:", style={"description_width": "70px"},
    layout=widgets.Layout(width="380px"))

def _fit_tier(gap, val_r2):
    if val_r2 < 0:
        return "Severe — extrapolation collapsing", PALETTE["secondary"]
    elif gap > 0.1:
        return "Mild overfitting", "#F59F00"
    else:
        return "Good fit", PALETTE["accent"]

def render_boundary(change=None):
    d = degree_sl_w2.value
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("scaler", StandardScaler()),
        ("lr", LinearRegression()),
    ])
    pipe.fit(_xtr_w2b.reshape(-1, 1), _ytr_w2b)
    train_r2 = r2_score(_ytr_w2b, pipe.predict(_xtr_w2b.reshape(-1, 1)))
    val_r2 = r2_score(_yva_w2b, pipe.predict(_xva_w2b.reshape(-1, 1)))
    gap = train_r2 - val_r2

    x_line = np.linspace(_x_w2b.min(), _x_w2b.max(), 300)
    y_line = pipe.predict(x_line.reshape(-1, 1))

    tier_label, tier_color = _fit_tier(gap, val_r2)
    val_r2_str = f"{val_r2:.3f}" if abs(val_r2) < 100 else f"{val_r2:.2e}"

    fig = go.Figure(data=[
        go.Scatter(x=_xtr_w2b, y=_ytr_w2b, mode="markers",
            marker=dict(color=PALETTE["muted"], size=5, opacity=0.4), name="Training"),
        go.Scatter(x=_xva_w2b, y=_yva_w2b, mode="markers",
            marker=dict(color=PALETTE["secondary"], size=5, opacity=0.5), name="Validation"),
        go.Scatter(x=x_line, y=y_line, mode="lines",
            line=dict(color=PALETTE["primary"], width=3), name=f"Degree {d} fit"),
    ], layout=base_layout(
        title=f"Degree {d}  |  Train R²={train_r2:.3f}  Val R²={val_r2_str}",
        xaxis_title="MedInc", yaxis_title="Median House Value",
    ))
    fig.update_yaxes(range=[0, 5.5])
    fig.add_annotation(
        x=0.02, y=0.98, xref="paper", yref="paper", showarrow=False,
        text=f"<b>{tier_label}</b>",
        font=dict(size=12, color=tier_color), align="left",
    )
    with out_boundary:
        clear_output(wait=True)
        fig.show()

degree_sl_w2.observe(render_boundary, names="value")
display(VBox([degree_sl_w2, out_boundary]))
render_boundary()


---
## What's happening?

In the **Basis Function Explorer**, each new degree adds a new direction the fit curve can bend. Degree 1 is a straight line, degree 2 is a parabola, degree 3 adds a single inflection point, and so on. The normal equations find the best combination of all these basis functions for the given data.

In the **Overfitting Boundary Widget**, training performance keeps improving with every degree because the model gets more and more tools to trace the exact shape of the training data. Validation performance tells a different story: it holds up reasonably well through the low-to-mid degrees, then falls off a cliff at high degree — not just a gentle decline, but a genuine collapse, because a high-degree polynomial swings wildly near the sparse edges of the data range. That whipsaw behavior at the boundaries has a name: **Runge's phenomenon**. It's a distinct failure mode from "learning the noise" — it's what happens when a high-degree curve is forced through unevenly-spaced points and has nowhere to go but to oscillate.

The right degree is where validation error is minimized. That balance point varies by dataset, which is why you always select it using cross-validation rather than guessing — and why the real-world example below uses actual 5-fold cross-validation instead of a single lucky (or unlucky) train/validation split.

---
## Key hyperparameters

**`degree`** (default `2` in PolynomialFeatures) — the maximum polynomial degree. This is the most important hyperparameter. Select it using cross-validation, not by minimizing training error.

**`interaction_only`** (default `False`) — if `True`, only interaction terms ($x_i x_j$) are generated, not powers ($x_i^2$). Useful when you believe features interact but their individual curvatures are not important.

**`include_bias`** (default `True`) — whether to include the constant feature (column of ones). Leave this `True` when `fit_intercept=False` on the downstream `LinearRegression`; otherwise set it to `False`.

For the full list of hyperparameters, see the sklearn documentation:
[https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Captures nonlinear relationships without changing the algorithm | Overfits aggressively at high degrees |
| Still interpretable at degree 2 or 3 | Feature count grows exponentially with degree and number of features |
| Works within the linear regression framework — fast and exact solution | Coefficients become uninterpretable at high degrees |
| Cross-validation gives a principled way to choose degree | Sensitive to outliers (high-degree polynomials swing wildly near extreme points) |
| Pipeline-compatible: PolynomialFeatures + LinearRegression | Extrapolates very poorly outside the training data range |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| The scatter plot shows a clear curved relationship | You have many features (exponential feature growth makes it impractical) |
| You need an interpretable nonlinear model (degree 2 is still readable) | The relationship has many inflections (splines or tree-based models are better) |
| You want to stay in the linear model framework for speed and simplicity | You need to extrapolate beyond the training range |
| Degree 2 or 3 captures the curvature you see visually | You are already seeing overfitting at degree 2 (insufficient data for nonlinearity) |
| You are building interaction features deliberately | Your priority is automatic feature selection (use Lasso instead) |

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
_x_all = housing.data["MedInc"].values
_y_all = housing.target.values
# Same n=100 subsample rationale as the widget above: enough rows to be a real
# demo, few enough that a high-degree single-feature polynomial can actually overfit.
_idx = np.random.RandomState(42).choice(len(_x_all), 100, replace=False)
x, y = _x_all[_idx], _y_all[_idx]

# Held out ONCE, untouched until the very end — this is what makes the final
# number trustworthy instead of optimistic.
x_pool, x_test, y_pool, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

degrees = list(range(1, 11))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_means, cv_mins, cv_maxs = [], [], []

for d in degrees:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("sc", StandardScaler()),
        ("lr", LinearRegression()),
    ])
    scores = cross_val_score(pipe, x_pool.reshape(-1, 1), y_pool, cv=kf, scoring="r2")
    cv_means.append(scores.mean())
    cv_mins.append(scores.min())
    cv_maxs.append(scores.max())

best_d = degrees[int(np.argmax(cv_means))]

# Refit on the FULL cv pool at the selected degree, evaluate ONCE on the held-out test set
final_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=best_d, include_bias=False)),
    ("sc", StandardScaler()),
    ("lr", LinearRegression()),
])
final_pipe.fit(x_pool.reshape(-1, 1), y_pool)
test_r2 = r2_score(y_test, final_pipe.predict(x_test.reshape(-1, 1)))

print(f"Best degree by mean 5-fold CV R²: {best_d}  (CV R² = {cv_means[best_d - 1]:.3f})")
print(f"Held-out test R² at degree {best_d}: {test_r2:.3f}")
print()
print("The test set was never touched during degree selection - this is the number")
print("that actually reflects how the model will perform on new data.")

_floor = -1.5
plot_means = [max(m, _floor) for m in cv_means]
plot_mins  = [max(m, _floor) for m in cv_mins]
plot_maxs  = [max(m, _floor) for m in cv_maxs]

fig = go.Figure(data=[
    go.Scatter(
        x=degrees + degrees[::-1],
        y=plot_maxs + plot_mins[::-1],
        fill="toself", fillcolor="rgba(150,150,150,0.25)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Fold min-max range",
    ),
    go.Scatter(
        x=degrees, y=plot_means, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2.5), marker=dict(size=8),
        name="Mean 5-fold CV R²",
    ),
    go.Scatter(
        x=[best_d], y=[plot_means[best_d - 1]], mode="markers",
        marker=dict(color=PALETTE["accent"], size=14, symbol="star"),
        name=f"Selected degree = {best_d}",
    ),
], layout=base_layout(
    title="Polynomial Degree vs Mean 5-Fold CV R² (values below -1.5 clipped for readability)",
    xaxis_title="Polynomial Degree",
    yaxis_title="R² (clipped at -1.5)",
))
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1), yaxis=dict(range=[-1.6, 1.05]))
fig.show()


---
## Real-world example: choosing degree with cross-validation, not guesswork

Using MedInc alone on a 100-row sample, the chart above runs genuine 5-fold cross-validation for degrees 1 through 10, then refits the selected degree on the full cross-validation pool and checks it exactly once against a held-out test set that was carved off before any of the degree selection happened.

- **Notice:** mean CV R² is fairly flat through degrees 1-5, with the shaded fold range showing real fold-to-fold variability even at low degree — a reminder that a single train/validation split (as opposed to 5 folds) can easily land on an unrepresentative number
- **Notice:** from around degree 6 onward, mean CV R² drops sharply and then goes deeply negative — this is Runge's phenomenon compounding across folds, not a gentle decline
- **Notice:** the selected degree (starred) is chosen purely from the cross-validation curve, before the held-out test set is touched at all

> **Discussion question:** the code above only checks the held-out test set once, after degree selection is already finished. What would happen to the trustworthiness of the final test R² if you instead tried several degrees directly against the test set and reported whichever looked best?

### Degree selection strategy

| Degree | Typical behavior | When to use |
|---|---|---|
| 1 | Straight line (linear) | When scatter plot is linear |
| 2 | Parabola (one bend) | Accelerating or decelerating trends |
| 3 | One inflection point | S-curves, growth-plateau patterns |
| 4+ | Multiple inflections | Usually overfitting; use cross-validation carefully |

> **Polynomial regression adds curves to the linear model by engineering powers of the input features — powerful and interpretable at low degree, but it overfits quickly at high degree and always requires cross-validation to choose the right one.**

---
*Next up: 04 — Ridge and Lasso Regression, which add a penalty to prevent overfitting without sacrificing the linear framework*